In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#  Only change from gap version: label_smoothing → 0.0 (recommended ablation)
#  Everything else is identical so the scaffold comparison is clean.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    "num_classes":      46,
    "image_size":       32,
    "batch_size":       100,
    "epochs":           50,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.0,      # removed — let the ablation be clean
    "val_split":        0.1,
    "data_dir":         "/kaggle/input/datasets/rautranjit/devanagari/DevanagariHandwrittenCharacterDataset",
    "results_dir":      "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)
NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE  (identical to gap version)
# ─────────────────────────────────────────────────────────────────────────────
train_full = keras.utils.image_dataset_from_directory(
    os.path.join(CFG["data_dir"], "Train"),
    image_size=(IMG, IMG), batch_size=None,
    color_mode="grayscale", label_mode="int", seed=SEED,
)
test_ds_raw = keras.utils.image_dataset_from_directory(
    os.path.join(CFG["data_dir"], "Test"),
    image_size=(IMG, IMG), batch_size=None,
    color_mode="grayscale", label_mode="int", seed=SEED,
)
total   = tf.data.experimental.cardinality(train_full).numpy()
n_val   = max(1, int(total * CFG["val_split"]))
n_train = total - n_val
train_ds_raw = train_full.take(n_train)
val_ds_raw   = train_full.skip(n_train)

def normalise(img, lbl):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

train_ds = (
    train_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: (batched)")

# ─────────────────────────────────────────────────────────────────────────────
#  3. BUILDING BLOCKS  (identical to gap version)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """Identical to gap version: 2 residual blocks + dense concat + depthwise downsample."""
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    cat = layers.Concatenate()([r1, r2])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

# ─────────────────────────────────────────────────────────────────────────────
#  4. MODEL — NO SCAFFOLD ABLATION
#
#  What changed vs gap version:
#    REMOVED  – 1×5 horizontal stroke scaffold conv path in stem
#    REMOVED  – channel_attention on dual-path concatenation (single path now)
#    REMOVED  – scaffold projection + AveragePooling at enc1, enc2, enc3
#    REMOVED  – three Add() scaffold injections
#    KEPT     – everything else: dense_res_block, multi-scale GAP, head
#
#  The stem is now a single 3×3 conv → BN → GELU → 1×1 projection,
#  exactly like a standard CNN backbone.
# ─────────────────────────────────────────────────────────────────────────────
def build_no_scaffold_net(
    num_classes:  int   = 46,
    image_size:   int   = 32,
    dropout_rate: float = 0.3,
    head_units:   int   = 256,
) -> Model:
    """
    No-scaffold ablation of whatnet-gap.

    Stem: single 3×3 conv path only (no 1×5 stroke scaffold).
    Encoder: 3× dense_res_block, NO scaffold residual injections.
    Head: multi-scale GAP fusion → Dense 256 → LayerNorm → GELU → Dropout → logits.

    Everything except the scaffold is bit-for-bit identical to whatnet-gap
    so any accuracy difference is attributable solely to the scaffold.
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem: single path (no scaffold) ──────────────────────────────────
    x    = layers.Conv2D(32, 3, padding="same", use_bias=False, name="stem_conv")(inp)
    x    = layers.BatchNormalization(name="stem_bn")(x)
    x    = layers.Activation(gelu, name="stem_act")(x)
    # Project to 64ch to match gap version's stem output width
    stem = layers.Conv2D(64, 1, use_bias=False, name="stem_proj")(x)
    stem = layers.BatchNormalization(name="stem_proj_bn")(stem)
    stem = layers.Activation(gelu, name="stem_proj_act")(stem)

    # ── Encoder: identical blocks, NO scaffold injections ────────────────
    enc1 = dense_res_block(stem, 64,  64)   # 16×16
    enc2 = dense_res_block(enc1, 64,  128)  #  8× 8
    enc3 = dense_res_block(enc2, 128, 256)  #  4× 4

    # ── Multi-scale GAP fusion (identical to gap version) ─────────────────
    gap1  = layers.GlobalAveragePooling2D(name="gap1")(enc1)   # 64-d
    gap2  = layers.GlobalAveragePooling2D(name="gap2")(enc2)   # 128-d
    gap3  = layers.GlobalAveragePooling2D(name="gap3")(enc3)   # 256-d
    fused = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])  # 448-d

    # ── Classification head (identical to gap version) ────────────────────
    x = layers.Dense(head_units, use_bias=False, name="head_dense")(fused)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation("gelu", name="head_act")(x)
    x = layers.Dropout(dropout_rate, name="head_drop")(x)
    outputs = layers.Dense(num_classes, name="logits")(x)

    return keras.Model(inputs=inp, outputs=outputs, name="NoScaffold-Net")

# ─────────────────────────────────────────────────────────────────────────────
#  5. LR SCHEDULE  (identical to gap version)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)
    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)
    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  6. COMPILE
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],  # 0.0
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAIN
# ─────────────────────────────────────────────────────────────────────────────
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

model = build_no_scaffold_net(NUM_CLASSES, IMG)
model = compile_model(model, total_steps)
model.summary()

ckpt_path = os.path.join(CFG["results_dir"], "noscaffold_best.keras")
callbacks = [
    ModelCheckpoint(
        ckpt_path, monitor="val_accuracy",
        save_best_only=True, verbose=1,
    ),
    EarlyStopping(
        monitor="val_accuracy", patience=10,
        restore_best_weights=True, verbose=1,
    ),
]

print("\n▶ Training: NoScaffold-Net")
t0 = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CFG["epochs"],
    callbacks=callbacks,   # callbacks re-enabled — best checkpoint saved
    verbose=1,
)
elapsed  = time.time() - t0
best_val = max(history.history["val_accuracy"]) * 100.0
print(f"\n✔ Done: best val acc = {best_val:.2f}%  wall time = {elapsed:.0f}s")

# ─────────────────────────────────────────────────────────────────────────────
#  8. EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
test_acc = test_acc_raw * 100.0
macro_f1 = compute_macro_f1(model, test_ds)

print("\n" + "═"*52)
print(f"  NoScaffold-Net   test acc : {test_acc:.2f}%")
print(f"  NoScaffold-Net   macro F1 : {macro_f1:.2f}%")
print(f"  NoScaffold-Net   params   : {model.count_params():,}")
print(f"  NoScaffold-Net   test loss: {test_loss:.4f}")
print("═"*52)

# ── Reference numbers from whatnet-gap for direct comparison ──────────────
print("\n  ABLATION SUMMARY")
print("  ─────────────────────────────────────────────────")
print(f"  whatnet-gap  (with scaffold) : 99.79%  3,559,414 params")
print(f"  NoScaffold-Net (no scaffold) : {test_acc:.2f}%  {model.count_params():,} params")
delta = test_acc - 99.79
print(f"  Δ accuracy                   : {delta:+.2f}%")
print("  ─────────────────────────────────────────────────")
if delta < -0.2:
    print("  → Scaffold is doing significant work. Keep it.")
elif delta < 0:
    print("  → Scaffold helps marginally. Worth keeping for the inductive bias.")
else:
    print("  → Scaffold makes no difference. The encoder alone suffices.")

# ─────────────────────────────────────────────────────────────────────────────
#  9. PERSIST
# ─────────────────────────────────────────────────────────────────────────────
results = {
    "NoScaffold-Net": {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
}
with open(os.path.join(CFG["results_dir"], "noscaffold_results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(CFG["results_dir"], "noscaffold_history.json"), "w") as f:
    json.dump(
        {k: [float(v) for v in vals] for k, vals in history.history.items()},
        f, indent=2,
    )
print("[INFO] Results saved.")

2026-04-23 11:42:27.839723: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776944548.016773      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776944548.079136      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776944548.521754      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776944548.521793      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776944548.521808      55 computation_placer.cc:177] computation placer alr

Found 78200 files belonging to 46 classes.


I0000 00:00:1776944599.543757      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776944599.549982      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 13800 files belonging to 46 classes.
[INFO] Train: 70,380 | Val: 7,820 | Test: (batched)


Model: "NoScaffold-Net"

...

 Trainable params: 3,535,310 (13.49 MB)

 Non-trainable params: 6,336 (24.75 KB)


▶ Training: NoScaffold-Net
Epoch 1/50


I0000 00:00:1776944670.437478     132 service.cc:152] XLA service 0x7d383c003080 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776944670.437528     132 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776944670.437533     132 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776944672.967142     132 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776944689.702992     132 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.6433 - loss: 1.4097
Epoch 1: val_accuracy improved from -inf to 0.94501, saving model to ./results/noscaffold_best.keras
704/704 ━━━━━━━━━━━━━━━━━━━━ 131s 137ms/step - accuracy: 0.6436 - loss: 1.4085 - val_accuracy: 0.9450 - val_loss: 0.1839
Epoch 2/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.9750 - loss: 0.0904
Epoch 2: val_accuracy improved from 0.94501 to 0.97212, saving model to ./results/noscaffold_best.keras
704/704 ━━━━━━━━━━━━━━━━━━━━ 74s 102ms/step - accuracy: 0.9750 - loss: 0.0904 - val_accuracy: 0.9721 - val_loss: 0.0894
Epoch 3/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.9821 - loss: 0.0606
Epoch 3: val_accuracy improved from 0.97212 to 0.97775, saving model to ./results/noscaffold_best.keras
704/704 ━━━━━━━━━━━━━━━━━━━━ 74s 103ms/step - accuracy: 0.9821 - loss: 0.0606 - val_accuracy: 0.9777 - val_loss: 0.0642
Epoch 4/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.9873 - loss:

In [ ]:
#  9. FINAL TEST-SET EVALUATION
results = {}

for name, model in model.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc = test_acc_raw * 100.0
    macro_f1 = compute_macro_f1(model, test_ds)   # integer-label dataset
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }

# Comparison table
print_comparison_table(results)